# MediAI — Rule-Based Prototype (Using uploaded datasets)

This notebook is built specifically to use the datasets you uploaded:
- `/mnt/data/Final_data.csv` (the Life Style / user data)
- `/mnt/data/meal_metadata.csv` (meal / nutrition metadata)

It demonstrates: data loading, EDA, feature engineering, a rule-based recommendation engine that uses user lifestyle + meal metadata, and JSON output ready for web integration.

## 1) Setup: install minimal dependencies (runs in Colab / local)

In [ ]:

# Install minimal packages (comment out in local env if not needed)
!pip install --quiet pandas numpy rapidfuzz pytesseract pillow

# Tesseract is not required for this dataset-focused notebook, but left here if OCR steps are later added
# !apt-get update -qq && apt-get install -y -qq tesseract-ocr libtesseract-dev

print('Setup complete')

## 2) Load the uploaded datasets from /mnt/data and show basic info

In [ ]:

import pandas as pd, os
final_path = '/mnt/data/Final_data.csv'
meal_path = '/mnt/data/meal_metadata.csv'

print('Files present in /mnt/data:')
print(os.listdir('/mnt/data'))

# Load Final_data.csv
if os.path.exists(final_path):
    final_df = pd.read_csv(final_path)
    print('\nLoaded Final_data.csv with shape:', final_df.shape)
    display(final_df.head())
else:
    final_df = None
    print('\nFinal_data.csv not found at /mnt/data/Final_data.csv')

# Load meal_metadata.csv
if os.path.exists(meal_path):
    meal_df = pd.read_csv(meal_path)
    print('\nLoaded meal_metadata.csv with shape:', meal_df.shape)
    display(meal_df.head())
else:
    meal_df = None
    print('\nmeal_metadata.csv not found at /mnt/data/meal_metadata.csv')

## 3) Quick EDA & cleaning of the lifestyle dataset (`Final_data.csv`)
We create simplified features useful for rule-based recommendations: age_group, smoker_flag, alcohol_flag, exercise_level, diet_type, health_conditions (if present). Adjust column names if different.

In [ ]:

import numpy as np

if final_df is not None:
    print('\nColumns:', final_df.columns.tolist())
    df = final_df.copy()

    # Attempt to standardize common fields - use heuristics if column names differ
    # Look for plausible column names
    colnames = [c.lower() for c in df.columns]

    # Age
    age_col = None
    for cand in ['age','age_years','years']:
        for c in df.columns:
            if cand in c.lower():
                age_col = c; break
        if age_col: break
    if age_col:
        df['age'] = pd.to_numeric(df[age_col], errors='coerce')
    else:
        df['age'] = np.nan

    # Smoking - try to find a column
    smoke_col = None
    for c in df.columns:
        if 'smok' in c.lower():
            smoke_col = c; break
    if smoke_col:
        df['smoker_flag'] = df[smoke_col].astype(str).str.lower().isin(['yes','y','1','true','smoker','smokes'])
    else:
        df['smoker_flag'] = False

    # Alcohol - try to find column
    alc_col = None
    for c in df.columns:
        if 'alco' in c.lower() or 'drink' in c.lower():
            alc_col = c; break
    if alc_col:
        df['alcohol_flag'] = df[alc_col].astype(str).str.lower().isin(['yes','y','1','true','drink','drinks'])
    else:
        df['alcohol_flag'] = False

    # Exercise - attempt to map to 'none','low','moderate','high'
    ex_col = None
    for c in df.columns:
        if 'exer' in c.lower() or 'phys' in c.lower() or 'activity' in c.lower():
            ex_col = c; break
    def map_ex(val):
        if pd.isna(val): return 'unknown'
        s = str(val).lower()
        if any(x in s for x in ['none','no','0','never']): return 'none'
        if any(x in s for x in ['rare','low','sometimes','1']): return 'low'
        if any(x in s for x in ['moder','3','regular','often']): return 'moderate'
        if any(x in s for x in ['high','daily','intense','5']): return 'high'
        return 'unknown'
    if ex_col:
        df['exercise_level'] = df[ex_col].apply(map_ex)
    else:
        df['exercise_level'] = 'unknown'

    # Diet - try to detect vegetarian/non-vegetarian
    diet_col = None
    for c in df.columns:
        if 'diet' in c.lower() or 'veg' in c.lower():
            diet_col = c; break
    if diet_col:
        df['diet_type'] = df[diet_col].astype(str).str.lower().apply(lambda s: 'vegetarian' if 'veg' in s else ('non-vegetarian' if any(x in s for x in ['non','meat','fish','chicken']) else 'other'))
    else:
        df['diet_type'] = 'unknown'

    # Simple health conditions: aggregate columns that mention common diseases
    health_cols = [c for c in df.columns if any(k in c.lower() for k in ['diab','hypert','heart','stroke','asthma','cancer','renal'])]
    if health_cols:
        df['health_conditions'] = df[health_cols].apply(lambda row: [c for c in health_cols if pd.notna(row[c]) and str(row[c]).strip()!=''], axis=1)
    else:
        df['health_conditions'] = [[] for _ in range(len(df))]

    print('\nSample processed profile columns: age, smoker_flag, alcohol_flag, exercise_level, diet_type, health_conditions')
    display(df[['age','smoker_flag','alcohol_flag','exercise_level','diet_type']].head())
else:
    print('No lifestyle dataframe to process.')

## 4) Explore `meal_metadata.csv` — map meals to nutrients and common tags
This will be used to suggest diet items (include / avoid) tailored to profile + medicine rules.

In [ ]:

if meal_df is not None:
    print('Meal metadata columns:', meal_df.columns.tolist())
    # Example: assume meal_df has columns like 'meal_id','meal_name','ingredients','calories','protein','fat','carbs'
    # Normalize column names
    cols = [c.lower() for c in meal_df.columns]
    # Try to identify ingredient/text column
    ingredient_col = None
    for c in meal_df.columns:
        if any(k in c.lower() for k in ['ingredient','ingredients','ingr','items']):
            ingredient_col = c; break
    print('Detected ingredient column:', ingredient_col)
    display(meal_df.head())
else:
    print('No meal metadata loaded.')

## 5) Build rule-base using dataset insights
We'll create a rules engine that: given a user profile (sampled from Final_data.csv) and a list of medicines (you can manually provide for demo), produces diet include/avoid lists using meal metadata and generic medicine-class rules.

- The medicine-specific rules are simple mappings (e.g., Statins -> avoid grapefruit).
- Lifestyle rules derive from the dataset (e.g., smokers -> recommend quitting tips).

In [ ]:

# Simple medicine rules (extend as needed)
CLASS_RULES = {
    'Statin': {
        'alerts': ['Avoid grapefruit juice — may increase statin concentration.'],
        'diet_avoid_keywords': ['grapefruit','grapefruit juice'],
        'diet_include_keywords': ['leafy','whole grain','oats','barley']
    },
    'Antidiabetic': {
        'alerts': ['Monitor carbohydrate intake; avoid sugary drinks.'],
        'diet_avoid_keywords': ['sugar','sugar syrup','sugary','sweet'],
        'diet_include_keywords': ['fiber','vegetable','lentils','beans','protein']
    },
    'Analgesic': {
        'alerts': ['Avoid excessive alcohol while using analgesics.'],
        'diet_avoid_keywords': ['alcohol'],
        'diet_include_keywords': ['hydration','water','fluids']
    }
}

# Lifestyle rules driven by dataset features
LIFESTYLE_RULES_TEXT = {
    'smoker': 'Quitting smoking reduces medication interactions and improves recovery.',
    'alcohol': 'Avoid alcohol with many medications to reduce liver strain.',
    'low_exercise': 'Start light walking 15-20 minutes daily; gradual increase recommended.'
}

# Helper: find meal suggestions by keyword matching against meal metadata
def find_meals_by_keywords(meal_df, include_kw=[], avoid_kw=[], top_n=10):
    # lower-case searchable representation (try 'ingredients' or 'meal_name')
    if meal_df is None:
        return [], []
    search_text_col = None
    for c in meal_df.columns:
        if c.lower() in ['ingredients','ingredient','ingr','items','components']:
            search_text_col = c; break
    if not search_text_col:
        # fallback to first text column
        for c in meal_df.columns:
            if meal_df[c].dtype == object:
                search_text_col = c; break
    if not search_text_col:
        return [], []
    s = meal_df.copy()
    s['_search'] = s[search_text_col].astype(str).str.lower()
    include_matches = []
    avoid_matches = []
    for kw in include_kw:
        include_matches.extend(list(s[s['_search'].str.contains(kw.lower(), na=False)].head(top_n)['meal_name'].astype(str)))
    for kw in avoid_kw:
        avoid_matches.extend(list(s[s['_search'].str.contains(kw.lower(), na=False)].head(top_n)['meal_name'].astype(str)))
    # deduplicate and limit
    include_matches = list(dict.fromkeys(include_matches))[:top_n]
    avoid_matches = list(dict.fromkeys(avoid_matches))[:top_n]
    return include_matches, avoid_matches

# Example: show matches for fiber/leafy
if meal_df is not None:
    inc, avo = find_meals_by_keywords(meal_df, include_kw=['leafy','lentil','oat','barley'], avoid_kw=['grapefruit','sugar'])
    print('Include examples:', inc[:5])
    print('Avoid examples:', avo[:5])
else:
    print('Meal metadata not available for keyword matching demo.')

## 6) End-to-end demo function: assemble input profile + medicines -> JSON output
This function will: accept a `user_profile` (can be sampled from Final_data.csv) and `prescription_list` (list of medicine dicts), run rules, query meal metadata for food suggestions, and output the structured JSON schema.

In [ ]:

import json
from datetime import datetime

def generate_recs_from_profile(user_profile, prescription_list, df_meals):
    # prescription_list: [{'name':'Atorvastatin','class':'Statin'}, ...]
    rec = {'diet':{'include':[], 'avoid':[]}, 'exercise':None, 'lifestyle_tips':[], 'alerts':[]}
    # lifestyle-driven tips
    if user_profile.get('smoker_flag') :
        rec['lifestyle_tips'].append(LIFESTYLE_RULES_TEXT['smoker'])
    if user_profile.get('alcohol_flag'):
        rec['lifestyle_tips'].append(LIFESTYLE_RULES_TEXT['alcohol'])
    if user_profile.get('exercise_level') in ['none','low']:
        rec['lifestyle_tips'].append(LIFESTYLE_RULES_TEXT['low_exercise'])

    include_kw = []
    avoid_kw = []
    # medicines rules
    for med in prescription_list:
        mclass = med.get('class')
        mname = med.get('name')
        if mclass and mclass in CLASS_RULES:
            rules = CLASS_RULES[mclass]
            rec['alerts'].extend([{'medicine': mname, 'alert': a} for a in rules.get('alerts', [])])
            include_kw.extend(rules.get('diet_include_keywords', []))
            avoid_kw.extend(rules.get('diet_avoid_keywords', []))
    # use meal metadata to find meal names to include/avoid
    inc_meals, avo_meals = find_meals_by_keywords(df_meals, include_kw=include_kw, avoid_kw=avoid_kw, top_n=8)
    rec['diet']['include'] = inc_meals
    rec['diet']['avoid'] = avo_meals

    # simple exercise suggestion
    if user_profile.get('exercise_level') in ['none','low']:
        rec['exercise'] = 'Start with 15-20 minutes walking daily; monitor response.'
    else:
        rec['exercise'] = 'Maintain current exercise; avoid sudden intense activity when starting new meds.'

    return rec

def build_output_json(user_profile, prescription_list, recs):
    out = {
        'user_profile': user_profile,
        'prescription_analysis': prescription_list,
        'recommendations': recs,
        'metadata': {'processed_at': datetime.utcnow().isoformat() + 'Z', 'version':'v0.2.0'}
    }
    return json.dumps(out, indent=2)

# Demo: sample a user profile from final_df (first row) and demo prescription list
if final_df is not None:
    sample_profile = df.iloc[0].to_dict()
    # simplify sample_profile to include keys we use
    user_profile = {
        'age': sample_profile.get('age'),
        'smoker_flag': sample_profile.get('smoker_flag', False),
        'alcohol_flag': sample_profile.get('alcohol_flag', False),
        'exercise_level': sample_profile.get('exercise_level', 'unknown'),
        'diet_type': sample_profile.get('diet_type', 'unknown'),
        'health_conditions': sample_profile.get('health_conditions', [])
    }
else:
    # fallback sample
    user_profile = {'age':45,'smoker_flag':False,'alcohol_flag':True,'exercise_level':'low','diet_type':'non-vegetarian','health_conditions':['Type 2 Diabetes']}

# Example prescription list (you can replace with normalized output later)
prescription_list = [{'name':'Atorvastatin','class':'Statin'},{'name':'Metformin','class':'Antidiabetic'}]

recs = generate_recs_from_profile(user_profile, prescription_list, meal_df)
out_json = build_output_json(user_profile, prescription_list, recs)
print(out_json[:1000])  # print truncated for demo

# Save JSON
out_path = '/mnt/data/med_output_from_dataset.json'
with open(out_path, 'w') as f:
    f.write(out_json)
print('\nSaved output JSON to', out_path)

## 7) Next steps (recommended)
- Expand CLASS_RULES with more drug classes and specific drug-level rules.
- Improve meal metadata mapping by adding nutrient columns and building scoring (e.g., high-fiber score).
- Integrate prescription OCR & normalization (previous notebook had OCR pipeline) to produce `prescription_list` automatically.
- Wrap this logic into an API (FastAPI) that accepts image + user_profile and returns this JSON.